In [1]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [2]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Couturier2020_Part2.h5ad")

In [3]:
adata = adata[adata.obs['donor_id'].isin(["BT346", "BT364", "BT389", "BT390", "BT397", "BT402", "BT407", "BT409"])]

In [4]:
df_obs = pd.DataFrame(adata.obs)

In [5]:
del adata.obs

In [6]:
adata = adata.raw.to_adata()

In [7]:
adata

AnnData object with n_obs × n_vars = 46250 × 16507
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [8]:
#adata = adata.raw.to_adata()

In [9]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 6250/6250 [00:07<00:00, 813.88it/s]


In [10]:
adata.X = X_counts_recovered

In [11]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [12]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [13]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [14]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [15]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF367,False,1023,2.211892,True,0.013086,0.009935,0.717126
SULT1B1,False,162,0.350270,True,0.002504,0.002231,1.079618
TRIM63,False,78,0.168649,True,0.001294,0.001196,0.948573
HDHD2,False,8507,18.393514,True,0.114070,0.075478,0.649020
MORF4L2-AS1,False,258,0.557838,True,0.003608,0.002922,0.836632
...,...,...,...,...,...,...,...
LINC02210-CRHR1,False,320,0.691892,True,0.004016,0.002906,0.749985
TPTEP2-CSNK1E,False,497,1.074595,True,0.007079,0.005984,0.911348
FAM237B,False,231,0.499459,True,0.003427,0.003572,0.942396


In [16]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [17]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [18]:
adata.var = df_tmp

In [19]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [20]:
adata

View of AnnData object with n_obs × n_vars = 46250 × 15359
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [21]:
adata.obs['donor_id'] = df_obs['donor_id']

In [22]:
# Create a DataFrame with the metadata for Couturier2020
metadata_data = {
    'Author': ['Couturier2020'] * 8,
    'donor_id': ["BT346", "BT364", "BT389", "BT390", "BT397", "BT402", "BT407", "BT409"],
    'stage': ['Primary'] * 8,
    'assay': ['10x 3\' v2'] * 8,
    'tissue': ['forebrain', 'forebrain', 'forebrain', 'forebrain', 'left frontal lobe', 'forebrain', 'forebrain', 'forebrain'],
    'Cells': ['Total'] * 8,
    'Method': ['cell'] * 8
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)

          Author donor_id    stage      assay             tissue  Cells Method
0  Couturier2020    BT346  Primary  10x 3' v2          forebrain  Total   cell
1  Couturier2020    BT364  Primary  10x 3' v2          forebrain  Total   cell
2  Couturier2020    BT389  Primary  10x 3' v2          forebrain  Total   cell
3  Couturier2020    BT390  Primary  10x 3' v2          forebrain  Total   cell
4  Couturier2020    BT397  Primary  10x 3' v2  left frontal lobe  Total   cell
5  Couturier2020    BT402  Primary  10x 3' v2          forebrain  Total   cell
6  Couturier2020    BT407  Primary  10x 3' v2          forebrain  Total   cell
7  Couturier2020    BT409  Primary  10x 3' v2          forebrain  Total   cell


In [23]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

      donor_id         Author    stage      assay     tissue  Cells Method
0        BT346  Couturier2020  Primary  10x 3' v2  forebrain  Total   cell
1        BT346  Couturier2020  Primary  10x 3' v2  forebrain  Total   cell
2        BT346  Couturier2020  Primary  10x 3' v2  forebrain  Total   cell
3        BT346  Couturier2020  Primary  10x 3' v2  forebrain  Total   cell
4        BT346  Couturier2020  Primary  10x 3' v2  forebrain  Total   cell
...        ...            ...      ...        ...        ...    ...    ...
46245    BT409  Couturier2020  Primary  10x 3' v2  forebrain  Total   cell
46246    BT409  Couturier2020  Primary  10x 3' v2  forebrain  Total   cell
46247    BT409  Couturier2020  Primary  10x 3' v2  forebrain  Total   cell
46248    BT409  Couturier2020  Primary  10x 3' v2  forebrain  Total   cell
46249    BT409  Couturier2020  Primary  10x 3' v2  forebrain  Total   cell

[46250 rows x 7 columns]


In [24]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [25]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
BT346_AAACCTGAGAGCTGCA-1-0,BT346,1357,1470.482422,Neoplastic,Differentiated-like,MES-like,Astrocyte,Fibroblasts,malignant cell
BT346_AAACCTGAGGTGCTTT-1-0,BT346,605,1039.133667,Neoplastic,Differentiated-like,MES-like,Astrocyte,Neural Stem/Precursor Cells,malignant cell
BT346_AAACCTGAGTACTTGC-1-0,BT346,2955,1958.033203,Neoplastic,Differentiated-like,MES-like,Astrocyte,Neural Stem/Precursor Cells,malignant cell
BT346_AAACCTGAGTATCTCG-1-0,BT346,970,1262.463257,Neoplastic,Differentiated-like,MES-like,Astrocyte,Neural Stem/Precursor Cells,malignant cell
BT346_AAACCTGAGTGGAGAA-1-0,BT346,719,1158.570435,Neoplastic,Differentiated-like,MES-like,Astrocyte,Neural Stem/Precursor Cells,malignant cell
...,...,...,...,...,...,...,...,...,...
BT409_TTTGTCACAATGAATG-1-0,BT409,736,1218.315430,Neoplastic,Differentiated-like,MES-like,Astrocyte,Fibroblasts,malignant cell
BT409_TTTGTCAGTACAGTTC-1-0,BT409,758,1359.908081,Neoplastic,Stem-like,OPC-like,Astrocyte,Pluripotent Stem Cells,malignant cell
BT409_TTTGTCAGTCAATGTC-1-0,BT409,792,1077.148682,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Monocytes,macrophage
BT409_TTTGTCAGTCCATCCT-1-0,BT409,2712,1742.403076,Non-neoplastic,Myeloid,TAM-BDM,Microglial cell,Microglia,macrophage


In [26]:
merged_obs_df.index= df_obs.index

In [27]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [28]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [29]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [30]:
del merged_obs_df['donor_id_y']

In [31]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [32]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [33]:
adata.obs = merged_obs_df

In [34]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    729 total control genes are used. (0:00:01)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    643 total control genes are used. (0:00:01)
-->     'phase', cell cycle phase (adata.obs)


In [35]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Couturier2020_Part3.h5ad")